In [44]:
import pandas as pd

long_df = pd.DataFrame(
    {
        "이름": ["철수", "철수", "영희", "영희"],
        "과목": ["수학", "영어", "수학", "영어"],
        "점수": [90, 85, 80, 95],
    }
)

print(long_df)
print()

wide_df = long_df.pivot(index="이름", columns="과목", values="점수")
print(wide_df)
print()

restored_df = wide_df.reset_index().melt(
    id_vars="이름",
    value_vars=["수학", "영어"],
    var_name="과목",
    value_name="점수",
)

print(restored_df)

   이름  과목  점수
0  철수  수학  90
1  철수  영어  85
2  영희  수학  80
3  영희  영어  95

과목  수학  영어
이름        
영희  80  95
철수  90  85

   이름  과목  점수
0  영희  수학  80
1  철수  수학  90
2  영희  영어  95
3  철수  영어  85


In [45]:
df = pd.DataFrame(
    {
        "수학": [90, 80],
        "영어": [85, 95],
    },
    index=["철수", "영희"],
)
print(df)
print()

stacked = df.stack()
print(stacked)
print()

unstacked = stacked.unstack()
print(unstacked)

    수학  영어
철수  90  85
영희  80  95

철수  수학    90
    영어    85
영희  수학    80
    영어    95
dtype: int64

    수학  영어
철수  90  85
영희  80  95


In [46]:
print(unstacked.T)

    철수  영희
수학  90  80
영어  85  95


In [47]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder

df_users = pd.DataFrame(
    {
        "ID": [101, 102, 103, 104, 105, 106],
        "Age": [23, np.nan, 34, 45, 45, 29],
        "Grade": ["Silver", "Gold", "Silver", "VIP", "VIP", "Gold"],
    }
)
df_orders = pd.DataFrame(
    {
        "ID": [101, 103, 104, 105, 106, 101],
        "Spent": [100, 150, 200, 250, 300, 120],
    }
)

df = pd.merge(df_users, df_orders, how="inner", on="ID")

print(df)
print()

df = df.drop_duplicates()
print(df)
print()

df["Age"] = df["Age"].fillna(df["Age"].median())

print(df)
print()

q1, q3 = df["Spent"].quantile([0.25, 0.75])
lower, upper = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
df["Spent"] = df["Spent"].clip(lower=lower, upper=upper)
print(df)
print()

df_target = df[(df["Age"] >= 30) | (df["Grade"] == "VIP")].copy()
print(df_target)
print()

df_target["Age_Group"] = pd.cut(
    df_target["Age"], bins=2, labels=["Young_Adult", "Senior_Adult"]
)
print(df_target)
print()

encoder = OrdinalEncoder(categories=[["Silver", "Gold", "VIP"]])
df_target["Grade_Enc"] = encoder.fit_transform(df_target[["Grade"]])

print(df_target)
print()
scaler = MinMaxScaler()
df_target["Spent_Scaled"] = scaler.fit_transform(df_target[["Spent"]])
print(df_target)
print()

final_df = df_target[["ID", "Age_Group", "Grade_Enc", "Spent_Scaled"]].reset_index(
    drop=True
)
print(final_df.round(3))

    ID   Age   Grade  Spent
0  101  23.0  Silver    100
1  101  23.0  Silver    120
2  103  34.0  Silver    150
3  104  45.0     VIP    200
4  105  45.0     VIP    250
5  106  29.0    Gold    300

    ID   Age   Grade  Spent
0  101  23.0  Silver    100
1  101  23.0  Silver    120
2  103  34.0  Silver    150
3  104  45.0     VIP    200
4  105  45.0     VIP    250
5  106  29.0    Gold    300

    ID   Age   Grade  Spent
0  101  23.0  Silver    100
1  101  23.0  Silver    120
2  103  34.0  Silver    150
3  104  45.0     VIP    200
4  105  45.0     VIP    250
5  106  29.0    Gold    300

    ID   Age   Grade  Spent
0  101  23.0  Silver    100
1  101  23.0  Silver    120
2  103  34.0  Silver    150
3  104  45.0     VIP    200
4  105  45.0     VIP    250
5  106  29.0    Gold    300

    ID   Age   Grade  Spent
2  103  34.0  Silver    150
3  104  45.0     VIP    200
4  105  45.0     VIP    250

    ID   Age   Grade  Spent     Age_Group
2  103  34.0  Silver    150   Young_Adult
3  104  45.0   

In [48]:
df_users = pd.DataFrame(
    {
        "Branch": ["Seoul", "Busan", "Seoul", "Daegu", "Busan", "Busan"],
        "Manager_Grade": ["Senior", "Junior", "Senior", "VIP", "Junior", "Junior"],
        "Score": [85, 72, 85, np.nan, 90, 65],
    }
)
df_sales = pd.DataFrame(
    {
        "Branch": ["Seoul", "Busan", "Daegu", "Gwangju"],
        "Jan_Sales": [50000, 75000, 999999, 60000],
        "Feb_Sales": [55000, 80000, 110000, 62000],
    }
)
print(df_users)
print()

df_users = df_users.drop_duplicates()
print(df_users)
print()

df_users["Score"] = df_users["Score"].fillna(df_users["Score"].median())
print(df_users)
print()

df_sales_long = df_sales.melt(
    id_vars=["Branch"],
    value_vars=["Jan_Sales", "Feb_Sales"],
    var_name="Month",
    value_name="Sales",
)
print(df_sales_long)
print()

df_merged = pd.merge(df_users, df_sales_long, on="Branch", how="inner")
df_filtered = df_merged[df_merged["Sales"] <= 100000].copy()

print(df_merged)
print()
print(df_filtered)
print()

df_pivot = df_filtered.pivot_table(
    index="Manager_Grade", columns="Month", values="Sales", aggfunc="mean"
).reset_index()
print(df_pivot)
print()

df_pivot["Sales_Level"] = pd.cut(df_pivot["Jan_Sales"], bins=2, labels=["Low", "High"])
print(df_pivot)
print()

encoder = OrdinalEncoder(categories=[["Junior", "Senior", "VIP"]])
df_pivot["Grade_Enc"] = encoder.fit_transform(df_pivot[["Manager_Grade"]])
print(df_pivot)
print()

scaler = MinMaxScaler()
df_pivot["Feb_Sales_Scaled"] = scaler.fit_transform(df_pivot[["Feb_Sales"]])
print(df_pivot)
print()

final_df = df_pivot[["Manager_Grade", "Grade_Enc", "Sales_Level", "Feb_Sales_Scaled"]]
print(final_df.round(3))

  Branch Manager_Grade  Score
0  Seoul        Senior   85.0
1  Busan        Junior   72.0
2  Seoul        Senior   85.0
3  Daegu           VIP    NaN
4  Busan        Junior   90.0
5  Busan        Junior   65.0

  Branch Manager_Grade  Score
0  Seoul        Senior   85.0
1  Busan        Junior   72.0
3  Daegu           VIP    NaN
4  Busan        Junior   90.0
5  Busan        Junior   65.0

  Branch Manager_Grade  Score
0  Seoul        Senior   85.0
1  Busan        Junior   72.0
3  Daegu           VIP   78.5
4  Busan        Junior   90.0
5  Busan        Junior   65.0

    Branch      Month   Sales
0    Seoul  Jan_Sales   50000
1    Busan  Jan_Sales   75000
2    Daegu  Jan_Sales  999999
3  Gwangju  Jan_Sales   60000
4    Seoul  Feb_Sales   55000
5    Busan  Feb_Sales   80000
6    Daegu  Feb_Sales  110000
7  Gwangju  Feb_Sales   62000

  Branch Manager_Grade  Score      Month   Sales
0  Seoul        Senior   85.0  Jan_Sales   50000
1  Seoul        Senior   85.0  Feb_Sales   55000
2  Busan 